# Evo Aggregated Statistics to MongoDB

This notebook calculates aggregated statistics from Evo downhole objects using the MCP data analysis utilities (shared with the tools) and stores them in a MongoDB collection. 

**Objectives**: 
- Test a basic implementation of the Evo MCP utilities > mongo DB integration
- Experiment with the calculated statistics to determine what is useful
- Assess performance on querying those statistics over a number of objects

**Steps:**
- Connects to Evo platform via hijacked OAuth token 
- Calculates interval statistics (length-weighted mean, accumulation, etc.)
- Calculates per-hole statistics
- Stores results with timestamps in MongoDB for tracking
- Analyzes the performance with and without indexing

**Prerequisites:**
- MongoDB running locally 
- Evo MCP configured with valid credentials in `.env`
- `pymongo` installed

#### Setup

In [1]:
import sys
import pandas as pd
import json
from pathlib import Path
from uuid import UUID

# Add src directory to path for imports
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Add notebooks directory to path for supporting_scripts
notebooks_path = Path.cwd()
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

# MongoDB utilities
from supporting_scripts.mongo_utils import (
    connect_to_mongodb,
    estimate_doc_size,
    build_stats_summary,
    build_gap_summary,
    build_categorical_summary,
    prepare_collection_documents,
    create_grade_stats_indexes,
    find_high_grade_objects,
    get_top_objects_by_grade,
    MONGO_DOC_LIMIT,
    SAFE_DOC_LIMIT,
)

# Benchmarking utilities
from supporting_scripts.stats_benchmark import (
    profile_query,
    get_explain_stats,
    run_index_benchmark,
    profile_high_grade_queries,
)

# Data loading utilities
from supporting_scripts.data_loading import download_all_interval_tables
from evo_mcp.utils.evo_data_utils import discover_objects, load_downhole_object

# Evo MCP utilities
from evo_mcp.context import evo_context, ensure_initialized
from evo_mcp.utils.data_analysis_utils import (
    calculate_interval_statistics,
    calculate_statistics_by_hole,
    calculate_categorical_statistics,
    analyze_gaps,
    calculate_multi_grade_statistics,
)

#### MongoDB onfiguration

Define the MongoDB connection settings and Evo workspace/object parameters.

In [5]:
# Load MongoDB connection parameters from config file

config_path = Path.cwd() / "mongo_config2.json"
if config_path.exists():
    with open(config_path, 'r') as f:
        mongo_config = json.load(f)
    
    # Check if Atlas credentials are provided
    if "username" in mongo_config and "password" in mongo_config and "cluster_url" in mongo_config:
        protocol = mongo_config.get("protocol", "mongodb+srv")
        username = mongo_config["username"]
        password = mongo_config["password"]
        cluster_url = mongo_config["cluster_url"]
        MONGO_URI = f"{protocol}://{username}:{password}@{cluster_url}"
        print(f"Using MongoDB Atlas ({protocol}): {cluster_url.split('/')[0]}")
    else:
        # Fall back to local MongoDB
        protocol = mongo_config.get("protocol", "mongodb")
        host = mongo_config.get("host", "localhost")
        port = mongo_config.get("port", 27017)
        MONGO_URI = f"{protocol}://{host}:{port}/"
        print(f"Using local MongoDB: {host}:{port}")
    
    MONGO_DB_NAME = mongo_config.get("database", "evo")
    MONGO_COLLECTION_NAME = mongo_config.get("collection", "interval_statistics")
else:
    print(f"Config file not found: {config_path}")
    MONGO_URI = "mongodb://localhost:27017/"
    MONGO_DB_NAME = "evo"
    MONGO_COLLECTION_NAME = "interval_statistics"

WORKSPACE_ID = "01c54ab3-0b97-4b36-8e72-686e65a906ed"

# Optional: specific version (leave empty for latest)
VERSION = ""

# If False, skip objects that already have documents in the collection
OVERWRITE_STATS = False

print(f"MongoDB: {MONGO_URI.split('@')[-1] if '@' in MONGO_URI else MONGO_URI}")
print(f"Database: {MONGO_DB_NAME}.{MONGO_COLLECTION_NAME}")
print(f"Overwrite: {OVERWRITE_STATS}")
print(f"Workspace: {WORKSPACE_ID}")

Config file not found: c:\Dev\evo-mcp-fork\notebooks\mongo_config2.json
MongoDB: mongodb://localhost:27017/
Database: evo.interval_statistics
Overwrite: False
Workspace: 01c54ab3-0b97-4b36-8e72-686e65a906ed


## 3. Connect to MongoDB

Establish connection to MongoDB and create/access the target collection.

Make sure you've built the docker image for the server and that is running before executing this cell.The docker command to run the server is:

```bash
docker run -d -p 27017:27017 --name mongodb mongo:latest
```

In [6]:
mongo_client, mongo_db, stats_collection = connect_to_mongodb(MONGO_URI, MONGO_DB_NAME, MONGO_COLLECTION_NAME)

✓ Connected to MongoDB: evo.interval_statistics


In [7]:
# Query existing data in the collection
doc_count = stats_collection.count_documents({})
print(f"Collection '{MONGO_COLLECTION_NAME}' has {doc_count} document(s)\n")

if doc_count > 0:
    # Distinct objects already stored
    stored_objects = stats_collection.distinct("object_id")
    print(f"Objects stored: {len(stored_objects)}")

    # Breakdown by object
    pipeline = [
        {"$group": {
            "_id": {"object_id": "$object_id", "object_name": "$object_name", "object_type": "$object_type"},
            "doc_count": {"$sum": 1},
            "collections": {"$addToSet": "$collection_name"},
            "latest": {"$max": "$timestamp"},
        }},
        {"$sort": {"_id.object_name": 1}},
    ]
    for row in stats_collection.aggregate(pipeline):
        info = row["_id"]
        print(f"\n  {info.get('object_name', '?')} ({info.get('object_type', '?')})")
        print(f"    ID: {info.get('object_id', '?')}")
        print(f"    Documents: {row['doc_count']}, Collections: {row['collections']}")
        print(f"    Latest: {row['latest']}")
else:
    print("Collection is empty — no data has been inserted yet.")

Collection 'interval_statistics' has 0 document(s)

Collection is empty — no data has been inserted yet.


## 4. Initialize Evo Connection

Initialize the Evo SDK context and authenticate via OAuth.

In [8]:
# Initialize Evo SDK connection (will trigger OAuth flow if needed)
await ensure_initialized()
print("Evo SDK initialized and authenticated")

Evo SDK initialized and authenticated


## 5. Load Object and Inspect Collections

Download the Evo object and inspect available collections/attributes.

In [9]:
# Discover all objects in the workspace, filtered to downhole types
all_objects = await discover_objects(
    WORKSPACE_ID,
    object_types=["downhole-collection", "downhole-intervals"],
)

print(f"Found {len(all_objects)} object(s) matching types ['downhole-collection', 'downhole-intervals']")
for o in all_objects:
    print(f"  {o['name']} ({o['schema_id']}) — {o['id']}")

Found 2 object(s) matching types ['downhole-collection', 'downhole-intervals']
  maia.json (downhole-collection) — 0286ea01-1a2c-41a8-81b8-fca7df38feac
  maia_geology_desurveyed.json (downhole-intervals) — ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e


## 6. Process All Objects → MongoDB

For each discovered object:
1. **Load** the object and inspect its interval collections
2. **Download** all interval tables, classifying columns as numeric vs categorical
3. **Calculate** statistics (overall + per-hole) and gap analysis for every numeric attribute
4. **Prepare** MongoDB documents (splitting large docs to stay under the 14 MB safe limit)
5. **Insert** into MongoDB

In [ ]:
import time

total_docs_inserted = 0
skipped_objects = []
failed_objects = []
inserted_object_ids = []  # Cache IDs of all successfully processed objects

# Pre-fetch existing object IDs from the collection
existing_object_ids = set(stats_collection.distinct("object_id")) if not OVERWRITE_STATS else set()

for obj_idx, obj_meta in enumerate(all_objects, 1):
    obj_id = obj_meta["id"]
    obj_name = obj_meta["name"]
    
    print(f"\n{'#'*70}")
    print(f"[{obj_idx}/{len(all_objects)}] {obj_name} ({obj_id})")
    print(f"{'#'*70}")
    
    # Skip if already in the database and not overwriting
    if obj_id in existing_object_ids:
        print(f"  Skipped (already in database)")
        skipped_objects.append(obj_name)
        inserted_object_ids.append(obj_id)  # Still track — data exists in DB
        continue
    
    try:
        # --- 1. Load object ---
        obj, obj_dict, object_name, object_type, collections_info = await load_downhole_object(
            WORKSPACE_ID, obj_id, VERSION
        )
        print(f"  Type: {object_type}, {len(collections_info)} interval table(s)")
        for coll in collections_info:
            attrs = coll.get("attributes", [])
            print(f"    {coll['name']} ({len(attrs)} attributes)")

        # --- 2. Download interval tables ---
        collection_data = await download_all_interval_tables(obj, object_type, collections_info)
        
        for coll_name, coll_data in collection_data.items():
            df = coll_data["df"]
            elapsed = coll_data["elapsed"]
            print(f"  {coll_name}: {len(df):,} intervals, "
                  f"{len(coll_data['numeric_cols'])} numeric / {len(coll_data['categorical_cols'])} categorical cols "
                  f"({elapsed:.1f}s)")

        # --- 3. Calculate statistics ---
        all_collection_stats = {}
        t0 = time.perf_counter()
        
        for coll_name, coll_data in collection_data.items():
            df = coll_data["df"]
            numeric_cols = coll_data["numeric_cols"]
            categorical_cols = coll_data["categorical_cols"]
            attribute_stats = {}
            
            # 3a. Numeric attribute statistics (LWM, accumulation, per-hole, etc.)
            for grade_col in numeric_cols:
                try:
                    overall = calculate_interval_statistics(df, grade_col)
                except (ValueError, ZeroDivisionError):
                    continue
                
                hole_stats_df = calculate_statistics_by_hole(df, grade_col)
                attribute_stats[grade_col] = {
                    "overall": overall,
                    "by_hole": hole_stats_df.to_dict(orient="records"),
                    "hole_count": len(hole_stats_df),
                }
            
            # 3b. Categorical attribute statistics (value counts, per-hole breakdown)
            categorical_stats = {}
            for cat_col in categorical_cols:
                try:
                    cat_stats = calculate_categorical_statistics(df, cat_col)
                    categorical_stats[cat_col] = cat_stats
                except (ValueError, Exception) as e:
                    print(f"    Warning: skipped categorical stat for '{cat_col}': {e}")
            
            gap_analysis = analyze_gaps(df)
            all_collection_stats[coll_name] = {
                "attributes": attribute_stats,
                "categorical": categorical_stats,
                "gap_analysis": gap_analysis,
            }
        
        stats_elapsed = time.perf_counter() - t0
        total_attrs = sum(len(v["attributes"]) for v in all_collection_stats.values())
        total_cats = sum(len(v["categorical"]) for v in all_collection_stats.values())
        print(f"  {total_attrs} numeric + {total_cats} categorical statistics across {len(all_collection_stats)} table(s) ({stats_elapsed:.1f}s)")

        # --- 4. Prepare MongoDB documents ---
        all_documents = []
        for coll_name, coll_stats in all_collection_stats.items():
            docs = prepare_collection_documents(
                workspace_id=WORKSPACE_ID,
                object_id=obj_id,
                object_name=object_name,
                object_type=object_type,
                collection_name=coll_name,
                attribute_stats=coll_stats["attributes"],
                gap_analysis=coll_stats["gap_analysis"],
                categorical_stats=coll_stats["categorical"],
            )
            all_documents.extend(docs)
        
        total_size_mb = sum(d["metadata"].get("doc_size_bytes", 0) for d in all_documents) / 1024 / 1024
        print(f"  {len(all_documents)} document(s), {total_size_mb:.2f} MB total")

        # --- 5. Insert into MongoDB ---
        if all_documents:
            result = stats_collection.insert_many(all_documents)
            total_docs_inserted += len(result.inserted_ids)
            inserted_object_ids.append(obj_id)
            print(f"  Inserted {len(result.inserted_ids)} document(s)")
        else:
            print(f"  No documents to insert")
    
    except Exception as e:
        failed_objects.append({"name": obj_name, "id": obj_id, "error": str(e)})
        print(f"  FAILED: {e}")

# --- Summary ---
print(f"\n{'='*70}")
print(f"Done: {len(all_objects)} objects, {total_docs_inserted} documents inserted, {len(skipped_objects)} skipped, {len(failed_objects)} failed")
print(f"Cached {len(inserted_object_ids)} object ID(s) for downstream queries")
if skipped_objects:
    print(f"\n{len(skipped_objects)} skipped (already in database):")
    for name in skipped_objects:
        print(f"  {name}")
if failed_objects:
    print(f"\n{len(failed_objects)} failed:")
    for f in failed_objects:
        print(f"  {f['name']}: {f['error']}")
if not OVERWRITE_STATS and skipped_objects:
    print(f"\nSet OVERWRITE_STATS = True to re-process skipped objects.")


######################################################################
[1/2] maia.json (0286ea01-1a2c-41a8-81b8-fca7df38feac)
######################################################################
  Type: downhole-collection, 2 interval table(s)
    assay (1 attributes)
    geology (1 attributes)
  assay: 2,210 intervals, 1 numeric / 0 categorical cols (11.7s)
  geology: 93 intervals, 0 numeric / 1 categorical cols (14.5s)
  1 attribute statistics across 2 table(s) (0.1s)
  2 document(s), 0.00 MB total
  Inserted 2 document(s)

######################################################################
[2/2] maia_geology_desurveyed.json (ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e)
######################################################################
  Type: downhole-intervals, 1 interval table(s)
    intervals (1 attributes)
  intervals: 93 intervals, 0 numeric / 1 categorical cols (9.8s)
  0 attribute statistics across 1 table(s) (0.0s)
  1 document(s), 0.00 MB total
  Inserted 1 document(s)



## 10. Create Grade Value Indexes

Create indexes to support queries like "find all high-grade objects" efficiently.

In [11]:
# Create the indexes
grade_indexes = create_grade_stats_indexes(stats_collection)

✓ Created 4 grade query indexes: ['grade_lwm', 'grade_accumulation', 'grade_max', 'workspace_grade_lwm']


## 10b. Index Performance Profiling

Compare query performance with and without indexes on the key query patterns.

In [12]:
# Run benchmark — documents are now present in the collection
doc_count = stats_collection.count_documents({})
print(f"Collection has {doc_count} documents")

if doc_count > 0:
    benchmark_results = run_index_benchmark(stats_collection)
else:
    print("No documents found — run the insert cell first")

Collection has 3 documents
Using sample: workspace_id=01c54ab3-0b97-4b36-8e72-686e65a906ed, object_id=0286ea01-1a2c-41a8-81b8-fca7df38feac

Testing index: workspace_object_idx
Query: {'workspace_id': '01c54ab3-0b97-4b36-8e72-686e65a906ed', 'object_id': '0286ea01-1a2c-41a8-81b8-fca7df38feac'}

WITHOUT INDEX:
   Mean: 1.1433 ms (±0.2693)
   Docs examined: 3
   Index used: workspace_grade_lwm

WITH INDEX:
   Mean: 1.1103 ms (±0.3263)
   Docs examined: 2
   Keys examined: 2
   Index used: workspace_object_idx

   Improvement: 2.9%

Testing index: object_name_idx
Query: {'object_name': {'$regex': '.*', '$options': 'i'}}

WITHOUT INDEX:
   Mean: 1.0620 ms (±0.2525)
   Docs examined: 3
   Index used: COLLSCAN

WITH INDEX:
   Mean: 1.0438 ms (±0.1542)
   Docs examined: 3
   Keys examined: 3
   Index used: object_name_idx

   Improvement: 1.7%


## 10. Verify and Query MongoDB

Query the collection to verify the data was stored correctly and explore historical statistics.

In [ ]:
# Collection overview
doc_count = stats_collection.count_documents({})
print(f"Total documents in collection: {doc_count}\n")

# Count by doc_type
for doc_type in ["complete", "summary", "detail"]:
    count = stats_collection.count_documents({"doc_type": doc_type})
    if count > 0:
        print(f"  {doc_type}: {count} document(s)")

# Show documents for all processed objects
for obj_id in inserted_object_ids:
    print(f"\n{'─'*60}")
    print(f"Documents for object {obj_id}:")
    cursor = stats_collection.find(
        {"object_id": obj_id},
        {
            "collection_name": 1, "doc_type": 1, "stats_summary": 1,
            "categorical_summary": 1, "gap_analysis": 1, "timestamp": 1,
            "metadata.doc_size_bytes": 1, "object_name": 1,
        }
    ).sort([("collection_name", 1), ("doc_type", 1)])

    for doc in cursor:
        coll = doc.get("collection_name", "?")
        dtype = doc.get("doc_type", "?")
        obj_name = doc.get("object_name", "?")
        size_kb = doc.get("metadata", {}).get("doc_size_bytes", 0) / 1024
        ts = doc.get("timestamp", "")
        
        if dtype in ("complete", "summary"):
            n_attrs = len(doc.get("stats_summary", []))
            n_cats = len(doc.get("categorical_summary", []))
            gaps = doc.get("gap_analysis", {}).get("total_gap_count", 0)
            print(f"\n  [{obj_name}] {coll} [{dtype}] - {n_attrs} numeric, {n_cats} categorical, {gaps} gaps, {size_kb:.0f} KB")
            
            # Show top 5 numeric attributes by LWM
            summary = sorted(doc.get("stats_summary", []), key=lambda s: abs(s.get("lwm") or 0), reverse=True)
            for s in summary[:5]:
                print(f"     {s['grade']}: LWM={s.get('lwm', 'N/A')}, max={s.get('max', 'N/A')}, n={s.get('count', 'N/A')}")
            if len(summary) > 5:
                print(f"     ... and {len(summary) - 5} more")
            
            # Show categorical attribute summaries
            cat_summary = doc.get("categorical_summary", [])
            if cat_summary:
                print(f"     Categorical attributes:")
                for cs in cat_summary:
                    top_vals = [v["value"] for v in cs.get("top_values", [])[:3]]
                    print(f"       {cs['attribute']}: {cs['unique_count']} unique values, "
                          f"top: {', '.join(top_vals)}")
        else:
            attrs = doc.get("attributes_in_chunk", [])
            print(f"\n  [{obj_name}] {coll} [{dtype}] - {len(attrs)} attributes (detail chunk), {size_kb:.0f} KB")

Total documents in collection: 3

  complete: 3 document(s)

────────────────────────────────────────────────────────────
Documents for object 0286ea01-1a2c-41a8-81b8-fca7df38feac:

  [Maia Drillholes] assay [complete] - 1 attributes, 0 gaps, 3 KB
     Au: LWM=0.4725068505663135, max=5.91, n=2210

  [Maia Drillholes] geology [complete] - 0 attributes, 0 gaps, 0 KB

────────────────────────────────────────────────────────────
Documents for object ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e:

  [Maia Geology (Desurveyed)] intervals [complete] - 0 attributes, 0 gaps, 0 KB


## 11. Query High-Grade Objects

Example queries for finding objects by grade statistics.

In [12]:
# Example queries - uncomment to run:

# 1. Find all objects with Au length-weighted mean >= 1.0 g/t
high_au_objects = find_high_grade_objects(stats_collection, grade="Au", min_lwm=0.4)
print(f"Found {len(high_au_objects)} objects with Au LWM >= 1.0")

# 2. Find objects with Au peak values >= 10 g/t
# high_peak_objects = find_high_grade_objects(stats_collection, grade="Au", min_max=10.0)

# 3. Top 10 objects by Au length-weighted mean
# top_au = get_top_objects_by_grade(stats_collection, grade="Au", metric="lwm", top_n=10)
# for obj in top_au:
#     print(f"  {obj['object_name']}: {obj['lwm']:.4f}")

# 4. Find high-grade objects in a specific workspace
# workspace_high_grade = find_high_grade_objects(
#     stats_collection, 
#     grade="Au", 
#     min_lwm=0.5, 
#     workspace_id=WORKSPACE_ID
# )

Found 1 objects with Au LWM >= 1.0


## 11b. Profile High-Grade Query Performance

Benchmark the high-grade query functions with and without indexes.

In [ ]:
# Run the profiling
doc_count = stats_collection.count_documents({})
print(f"Collection has {doc_count} documents\n")

if doc_count > 0:
    # Use first available grade column or default to "Au"
    test_grade = available_grade_columns[0] if 'available_grade_columns' in dir() and available_grade_columns else "Au"
    print(f"Profiling with grade: {test_grade}\n")
    grade_query_results = profile_high_grade_queries(stats_collection, grade=test_grade)
else:
    print("Insert documents first, then run this cell for profiling")

Collection has 2 documents

Profiling with grade: Au

DROPPING GRADE INDEXES FOR BASELINE...

📊 WITHOUT INDEXES:

   find_high_grade_objects(Au, min_lwm=0.5)
      Mean: 0.9092 ms (±0.1477)
      Docs examined: 2, Index: COLLSCAN

   find_high_grade_objects(Au, min_max=5.0)
      Mean: 0.9378 ms (±0.1358)
      Docs examined: 2, Index: COLLSCAN

   get_top_objects_by_grade(Au, lwm, top_n=10)
      Mean: 0.9223 ms (±0.1178)
      Docs examined: N/A (aggregate), Index: N/A (aggregate)


CREATING GRADE INDEXES...
Created: ['grade_lwm', 'grade_max', 'grade_accumulation', 'workspace_grade_lwm']

📊 WITH INDEXES:

   find_high_grade_objects(Au, min_lwm=0.5)
      Mean: 0.9141 ms (±0.2584)
      Docs examined: 0, Keys: 0, Index: grade_lwm

   find_high_grade_objects(Au, min_max=5.0)
      Mean: 0.8872 ms (±0.1418)
      Docs examined: 1, Keys: 1, Index: grade_max

   get_top_objects_by_grade(Au, lwm, top_n=10)
      Mean: 0.9517 ms (±0.1336)
      Docs examined: N/A, Keys: N/A, Index: N/A (agg

: 

: 